# Importar datos

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

# Configuración de rutas
data_path = r"C:\Users\xXSrBiscuitXx\Documents\GitHub\ProjecteData\Equip_15\Data\RRHH_220925_clean.parquet"

try:
    df = pd.read_parquet(data_path)
    print(f"✓ Dataset cargado correctamente: {df.shape[0]} filas, {df.shape[1]} columnas")
except Exception as e:
    print(f"✗ Error cargando el archivo: {e}")
    exit()

✓ Dataset cargado correctamente: 801 filas, 26 columnas


# Preparar variables


In [39]:
# Hit target average
pd.set_option("display.max.columns", None)
# Calculamos la media de hit target de cada trabajador y nos quedamos con un solo registro por trabajador 
df['Hit_target_avg'] = df.groupby('ID')['Hit_target'].transform('mean')

# Disciplinary failure
df["AtLeastOneDiscFailure"] = df.groupby("ID")["Disciplinary_failure"].transform(lambda x: int(x.max()))

# Suma de horas
df["TotalHours"] = df.groupby("ID")["Absenteeism_hours"].transform("sum")

# Conteo faltas injustificadas
df["UnjustifiedCount"] = df.groupby("ID")["Reason_absence"].transform(lambda x: (x == "Ausencia injustificada").sum())

In [43]:
df = df.drop_duplicates(subset=["ID"], keep="last").reset_index(drop=True)

df[df["UnjustifiedCount"]>0]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,Hit_target,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,Reason_absence_numeric,Education_numeric,Month_absence_order,Day_week_order,Seasons_order,Hit_target_avg,AtLeastOneDiscFailure,TotalHours,UnjustifiedCount
13,34,Fisioterapia,Enero,Lunes,Otono,118.0,10.0,10.0,37.0,308.593,95.0,0,High school,0,0,0,0,83.0,172.0,28.0,0.0,27,1,1,2,2,94.620000,0,333.0,3
17,33,No corresponde,Octubre,Viernes,Primavera,248.0,25.0,14.0,47.0,265.017,88.0,1,High school,2,0,0,1,86.0,165.0,32.0,0.0,0,1,10,6,4,94.791667,1,73.0,1
18,20,No corresponde,Octubre,Martes,Primavera,260.0,50.0,11.0,36.0,265.017,88.0,1,High school,4,1,0,0,65.0,168.0,23.0,0.0,0,1,10,3,4,94.571429,1,306.0,4
20,18,No corresponde,Noviembre,Martes,Primavera,330.0,16.0,4.0,28.0,284.031,97.0,1,Graduate,0,0,0,0,84.0,182.0,25.0,0.0,0,2,11,3,4,96.250000,1,118.0,2
22,13,No corresponde,Marzo,Miercoles,Otono,369.0,17.0,12.0,31.0,244.387,98.0,1,High school,3,1,0,0,70.0,169.0,25.0,0.0,0,1,3,4,2,92.866667,1,183.0,2
23,1,No corresponde,Marzo,Jueves,Verano,235.0,11.0,14.0,37.0,244.387,98.0,1,Postgraduate,1,0,0,1,88.0,172.0,29.0,0.0,0,3,3,5,3,95.173913,1,121.0,2
24,24,No corresponde,Marzo,Jueves,Verano,246.0,25.0,16.0,41.0,244.387,98.0,1,High school,0,1,0,0,67.0,170.0,23.0,0.0,0,1,3,5,3,94.142857,1,251.0,2
25,3,No corresponde,Junio,Viernes,Verano,179.0,51.0,18.0,38.0,253.957,95.0,1,High school,0,1,0,0,89.0,170.0,31.0,0.0,0,1,6,6,3,94.762887,1,444.0,1
28,11,No corresponde,Noviembre,Miercoles,Primavera,289.0,36.0,13.0,33.0,268.519,93.0,1,High school,2,1,0,1,90.0,172.0,30.0,0.0,0,1,11,4,4,93.850000,1,450.0,6
29,5,No corresponde,Noviembre,Jueves,Primavera,235.0,20.0,13.0,43.0,268.519,93.0,1,High school,1,1,0,0,106.0,167.0,38.0,0.0,0,1,11,5,4,91.722222,1,102.0,9


# Clustering

In [65]:
import pandas as pd
from kmodes.kprototypes import KPrototypes

# Crear dataframe solo con las columnas necesarias
dfClus = df[["Hit_target_avg", "AtLeastOneDiscFailure", "TotalHours", "UnjustifiedCount"]]

# Convertir a numpy
X = dfClus.to_numpy()

# Columnas categóricas (AtLeastOneDiscFailure está en la posición 1)
cat_cols = [1]

# Entrenar modelo
kproto = KPrototypes(n_clusters=4, random_state=42, init='Huang')
clusters = kproto.fit_predict(X, categorical=cat_cols)

# Guardar clusters (forzar a int para evitar errores)
dfClus["cluster"] = clusters.astype(int)

resumen = dfClus.groupby("cluster").agg({
    "Hit_target_avg": "mean",
    "AtLeastOneDiscFailure": "mean",   # será la proporción de 1s
    "TotalHours": "mean",
    "UnjustifiedCount": "mean",
    "cluster": "count"                 # cantidad de empleados por cluster
}).rename(columns={"cluster": "n_empleados"})

print(resumen)
print("\nCentroides:")

print(kproto.cluster_centroids_)
print(dfClus["cluster"].value_counts())

         Hit_target_avg  AtLeastOneDiscFailure  TotalHours  UnjustifiedCount  \
cluster                                                                        
0             94.528230               0.101852    7.879630          0.018519   
1             94.054263               0.500000  227.833333          0.666667   
2             95.110255               0.363636   88.636364          1.272727   
3             94.678562               0.714286  380.428571          2.142857   

         n_empleados  
cluster               
0                108  
1                  6  
2                 11  
3                  7  

Centroides:
[[9.45282297e+01 7.87962963e+00 1.85185185e-02 0.00000000e+00]
 [9.40542635e+01 2.27833333e+02 6.66666667e-01 0.00000000e+00]
 [9.51102547e+01 8.86363636e+01 1.27272727e+00 0.00000000e+00]
 [9.46785617e+01 3.80428571e+02 2.14285714e+00 1.00000000e+00]]
cluster
0    108
2     11
3      7
1      6
Name: count, dtype: int64


C:\Users\xXSrBiscuitXx\AppData\Local\Temp\ipykernel_26660\2263056077.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfClus["cluster"] = clusters.astype(int)


# Score Clustering

In [78]:
from sklearn.preprocessing import StandardScaler
from kmodes.kprototypes import KPrototypes

# Seleccionamos las columnas
dfClus = df[["Hit_target_avg", "AtLeastOneDiscFailure", "TotalHours", "UnjustifiedCount"]].copy()

# Escalamos solo las columnas numéricas
num_cols = ["Hit_target_avg", "TotalHours", "UnjustifiedCount"]
X_num = dfClus[num_cols].values
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

# Reconstruimos X agregando la columna categórica (falta disciplinaria)
X = np.hstack([X_num_scaled, dfClus[["AtLeastOneDiscFailure"]].values])

# Ahora la columna categórica es la última
cat_cols = [3]

# Entrenar K-Prototypes
kproto = KPrototypes(n_clusters=4, random_state=42, init='Huang')
clusters = kproto.fit_predict(X, categorical=cat_cols)

# Guardar clusters en el DataFrame
dfClus["cluster"] = clusters.astype(int)

resumen = dfClus.groupby("cluster").agg({
    "Hit_target_avg": "mean",
    "AtLeastOneDiscFailure": "mean",   # será la proporción de 1s
    "TotalHours": "mean",
    "UnjustifiedCount": "mean",
    "cluster": "count"                 # cantidad de empleados por cluster
}).rename(columns={"cluster": "n_empleados"})
print(resumen)

# Imprimir cost
print("Cost del clustering:", kproto.cost_)

# Ver la distribución de clusters
print(dfClus["cluster"].value_counts())

# Opcional: mostrar centroides
print("Centroides:")
print(kproto.cluster_centroids_)

         Hit_target_avg  AtLeastOneDiscFailure  TotalHours  UnjustifiedCount  \
cluster                                                                        
0             85.133333               0.300000    3.700000          0.100000   
1             93.381217               1.000000  286.000000          6.333333   
2             95.479077               0.084906   13.660377          0.018868   
3             94.621384               0.615385  270.230769          1.000000   

         n_empleados  
cluster               
0                 10  
1                  3  
2                106  
3                 13  
Cost del clustering: 117.04817604353042
cluster
2    106
3     13
0     10
1      3
Name: count, dtype: int64
Centroides:
[[-2.63620295 -0.42444982 -0.15303993  0.        ]
 [-0.33042409  2.5222169   5.62316453  1.        ]
 [ 0.25605375 -0.32048272 -0.22822209  0.        ]
 [ 0.01627721  2.35761659  0.6809575   1.        ]]
